# 第 17 章　線形回帰

明日の早朝と夕方に，同じ約 144 件の貸出を見込んでよいでしょうか。時刻によって予測値が変わる直線をデータに当てはめ，前章の一律平均と比べてみましょう。後半の約 20 分の実習では，自分で時間帯を選び，どちらの予測が実測に近かったかを報告します。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。ローカルでは配布 ZIP の配置で開くか，このノートと CSV を同じフォルダーに置きます。手元にファイルがなければ公開 URL から読み込みます。

## 講義　時刻から貸出件数を予測する

### 一律平均を思い出す

CSV の一行は一つの日付の一つの時間帯です。時刻 `hr` から，その時間帯の貸出件数 `cnt` を予測します。

前日の終了時に翌日の各時間帯を予測する設定です。2011 年の収録 8,645 行で規則を一度決め，2012 年 1〜6 月の収録 4,358 行に使います。春学期に調べた 2 年分のうち，2012 年後半は秋学期のモデル比較から保留します。時刻が欠けている日は，収録された時間帯を使います。

日付で学習用と検証用の行を選びます。一律平均を決める `mean()` に渡しているのはどちらの期間ですか。図の丸と破線の意味を区別して，早朝と夕方で予測をどちらへ動かすとよさそうかを考えます。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(csv_path, parse_dates=["dteday"])
train = hourly.loc[hourly["dteday"] < "2012-01-01"]
valid = hourly.loc[
    (hourly["dteday"] >= "2012-01-01") & (hourly["dteday"] < "2012-07-01")
]

baseline = train["cnt"].mean()
print(f"Training: {len(train):,} rows")
print(f"Validation: {len(valid):,} rows")
print(f"Baseline: {baseline:.2f} rentals per hour")

In [ ]:
plt.rcParams["figure.figsize"] = (7.2, 3.4)

actual_by_hour = valid.groupby("hr")["cnt"].mean()
fig, ax = plt.subplots(layout="constrained")
ax.scatter(actual_by_hour.index, actual_by_hour, color="#286b70", s=33,
           label="Observed hourly mean", zorder=3)
ax.axhline(baseline, color="#687581", linestyle="--", label="Constant baseline")
ax.set(xlabel="Hour of day", ylabel="Rentals per hour",
       xlim=(-0.5, 23.5), ylim=(0, 580), xticks=[0, 4, 8, 12, 17, 23])
ax.legend(frameon=False, loc="upper left")
plt.show()

### 三行で残差を見る

予測直線を $\hat y=a+bx$ と書きます。$a$ は切片，$b$ は傾きです。架空の三行です。0，1，2 時台の実測は順に 2，8，8 件とします。A は一律 6 件，B は時刻を $x$ として $3+3x$ 件を返します。図の時刻を一つ選び，実測から予測を引いた**残差**を読みます。その後に，A と B の各三行の残差を二乗して足すと，どちらが小さいかを計算してください。

さらに，0 時台と 2 時台の実測を結ぶ $2+3x$ 件の直線を考えます。二本の右上がりの直線を，二乗の合計と絶対値の平均で比べ，どちらが小さくなるかを確かめます。

第 $i$ 行の時刻を $x_i$，実測を $y_i$ とすると，予測は $\hat y_i=a+bx_i$ です。最小二乗法では，$n$ 行の残差平方和 $\mathrm{RSS}=\sum_{i=1}^n(y_i-\hat y_i)^2$ が最小になる切片 $a$ と傾き $b$ を求めます。この三行では A の RSS は 24，B は 6 です。各 RSS を 3 で割ると，平均二乗誤差（MSE）はそれぞれ 8 と 2 です。

In [ ]:
plt.rcParams["figure.figsize"] = (7.2, 3.35)

toy_hours = [0, 1, 2]
toy_actual = [2, 8, 8]
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, layout="constrained")
for ax, prediction, title in [
    (axes[0], [6, 6, 6], "A  Constant"),
    (axes[1], [3, 6, 9], "B  Rising line"),
]:
    ax.plot(toy_hours, prediction, color="#a34f30", marker="s", label="Prediction")
    ax.vlines(toy_hours, prediction, toy_actual, color="#687581", linestyle=":")
    ax.scatter(toy_hours, toy_actual, color="#286b70", label="Observed", zorder=3)
    ax.set(title=title, xlabel="Hour of day", xlim=(-0.25, 2.25), ylim=(0, 10.5),
           xticks=toy_hours)
axes[0].set_ylabel("Rentals per hour")
axes[0].legend(frameon=False, loc="lower right", fontsize=8)
plt.show()

### 時刻と件数の対で `fit()`

`X_train` の一行は，一つの時間帯の特徴量です。`y_train` の同じ位置に，その時間帯に実際にあった貸出件数があります。`X_train` と `y_train` の形を確かめてから，`fit()` で残差の二乗和が最小になる直線を求めます。切片は 0 時台の予測値です。傾きは，時刻の数字が 1 増えたときに予測値が何件変わるかを表します。

最後のセルで式に時刻を入れた値を確かめます。自分で 19 時台の予測を求めるなら，どの数字を替えますか。

In [ ]:
X_train = train[["hr"]]
y_train = train["cnt"]

print("X:", X_train.shape)
print("y:", y_train.shape)
train[["dteday", "hr", "cnt"]].head(3)

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

intercept = model.intercept_
slope = model.coef_[0]
print(f"Intercept: {intercept:.5f}")
print(f"Slope:     {slope:.5f}")

In [ ]:
print(f"00:00  {intercept + slope * 0:.2f}")
print(f"17:00  {intercept + slope * 17:.2f}")
print(f"18:00  {intercept + slope * 18:.2f}")

### 検証 MAE を一律平均と比べる

検証期間の時刻を `predict()` に渡します。検証期間の `cnt` は，最後に実測との誤差を求めるところで使います。予測を入れた表の先頭で，日付・時刻・実測と予測を読み，残差の符号を確かめます。続いて一行ずつの絶対誤差を平均し，全体の MAE を比べます。二つの平均には同じ 4,358 行を使います。

In [ ]:
X_valid = valid[["hr"]]
linear_prediction = model.predict(X_valid)

print(f"Predictions: {len(linear_prediction):,}")
print(linear_prediction[:3].round(2))

In [ ]:
evaluation = valid[["dteday", "hr", "cnt"]].copy()
evaluation["baseline_prediction"] = baseline
evaluation["linear_prediction"] = linear_prediction
evaluation["linear_residual"] = evaluation["cnt"] - evaluation["linear_prediction"]
evaluation["linear_abs_error"] = evaluation["linear_residual"].abs()
evaluation["baseline_abs_error"] = (
    evaluation["cnt"] - evaluation["baseline_prediction"]
).abs()

evaluation[["dteday", "hr", "cnt", "linear_prediction", "linear_residual"]].head(3)

In [ ]:
mae = evaluation[["baseline_abs_error", "linear_abs_error"]].mean()
print(f"Constant baseline: {mae['baseline_abs_error']:.2f} rentals per hour")
print(f"Linear regression: {mae['linear_abs_error']:.2f} rentals per hour")
print(f"Reduction:         {mae['baseline_abs_error'] - mae['linear_abs_error']:.2f}")

### 時刻ごとに誤差を調べる

実測と二つの予測が重なる図を見ます。4 時台，8 時台，17 時台で，一律平均から直線に移ると，予測はどこへ動きましたか。その時刻の MAE がどちらの方法で小さかったかを，次の表で確かめます。

In [ ]:
plt.rcParams["figure.figsize"] = (7.2, 4.2)

hourly_predictions = evaluation.groupby("hr")["linear_prediction"].mean()
fig, ax = plt.subplots(layout="constrained")
ax.scatter(evaluation["hr"], evaluation["cnt"], s=9, color="#738795",
           alpha=0.14, linewidths=0, label="Observed row")
ax.scatter(actual_by_hour.index, actual_by_hour, color="#286b70", s=31,
           label="Observed hourly mean", zorder=3)
ax.axhline(baseline, color="#687581", linestyle="--", label="Constant baseline")
ax.plot(hourly_predictions.index, hourly_predictions, color="#a34f30",
        linewidth=2, label="Linear prediction")
ax.set(xlabel="Hour of day", ylabel="Rentals per hour",
       xlim=(-0.5, 23.5), ylim=(0, 1000), xticks=[0, 4, 8, 12, 17, 23])
ax.legend(frameon=False, loc="upper left", fontsize=9)
plt.show()

In [ ]:
by_hour = evaluation.groupby("hr").agg(
    rows=("cnt", "size"),
    observed_mean=("cnt", "mean"),
    linear_prediction=("linear_prediction", "mean"),
    baseline_mae=("baseline_abs_error", "mean"),
    linear_mae=("linear_abs_error", "mean"),
    mean_residual=("linear_residual", "mean"),
)

by_hour.loc[[4, 8, 17, 23]].round(2)

8 時台の検証行から，実測が最も少なかった日と最も多かった日を取り出します。時刻が同じ二日で，直線の予測と絶対誤差をそれぞれ見比べます。続く図では，直線がどの時間帯にどちらの向きへ外れやすいかを，平均残差の点と 0 の線から読みます。

In [ ]:
at_eight = evaluation.loc[evaluation["hr"] == 8]
low_day = at_eight["cnt"].idxmin()
high_day = at_eight["cnt"].idxmax()

at_eight.loc[[low_day, high_day], [
    "dteday", "hr", "cnt", "linear_prediction", "linear_residual",
    "baseline_abs_error", "linear_abs_error",
]]

In [ ]:
plt.rcParams["figure.figsize"] = (7.2, 4)

fig, ax = plt.subplots(layout="constrained")
ax.scatter(evaluation["hr"], evaluation["linear_residual"], color="#738795",
           s=9, alpha=0.17, linewidths=0, label="Residual per row")
ax.scatter(by_hour.index, by_hour["mean_residual"], color="#a34f30", marker="D",
           s=28, zorder=3, label="Mean residual by hour")
ax.axhline(0, color="#687581", linestyle="--", linewidth=1)
ax.set(xlabel="Hour of day", ylabel="Observed minus predicted (rentals)",
       xlim=(-0.5, 23.5), xticks=[0, 4, 8, 12, 17, 23])
ax.legend(frameon=False, loc="upper left", fontsize=9)
plt.show()

## 実習　自分で選んだ時間帯の予測を報告する

### 1. 時間帯を選んで予想する（約 5 分）

講義で詳しく取り上げた時間帯とは別の時間帯を一つ選びます。たとえば 1 時台，7 時台，13 時台，21 時台などから選べます。担当者がその時間帯の貸出に備える理由と，直線に替えたときに誤差がどうなりそうかを，実習の表を作る前に書きます。

（選んだ時刻，その理由と予想をここに記入）

### 2. 行と誤差を確かめる（約 9 分）

`target_hour` の `None` を 0〜23 の整数に替えます。`if` の内側で `pass` の行を，`evaluation` からその `hr` の行だけを選んで `chosen` に入れる一行に替えます。書き方は，講義で 8 時台の行を取り出したコードを読み返せます。

入力ができると，選んだ時刻の行数・二つの MAE・直線の平均残差が表示されます。MAE を比べたあと，平均残差の符号から，直線が実際より多め・少なめのどちらに外れたかを読みます。

In [ ]:
target_hour = None
chosen = None

if isinstance(target_hour, int) and 0 <= target_hour <= 23:
    # evaluation から hr が target_hour の行を選び，chosen に入れる。
    pass

if chosen is None:
    print("時刻を指定し，その時刻の行を取り出すコードを書きます。")
elif chosen.empty:
    print("該当する行がありません。条件に指定した列と時刻を確かめます。")
else:
    print(f"{len(chosen):,} rows at {target_hour:02d}:00")
    print(f"Constant baseline MAE:  {chosen['baseline_abs_error'].mean():.2f}")
    print(f"Linear regression MAE:  {chosen['linear_abs_error'].mean():.2f}")
    print(f"Mean residual:          {chosen['linear_residual'].mean():+.2f}")
    display(chosen.head())

次は，日付の異なる二行を選びます。時刻は同じなので，予測はどちらの日にも同じです。まず `chosen` の表から実測の異なる日を探し，二つの日付を `selected_dates` のリストに文字列で入れます。日付の書き方は `"2012-06-01"` のように年・月・日の順です。別の日を探すときは，セルを追加して `chosen.tail()` などで表の別の行を表示できます。表に出た二行について，実測と二つの予測を見比べ，どちらが近かったかを日付ごとに読みます。

In [ ]:
selected_dates = []

if chosen is None or chosen.empty or len(selected_dates) != 2:
    print("時刻を選んだあと，異なる日付を二つ指定します。")
else:
    selected_rows = chosen.loc[chosen["dteday"].isin(pd.to_datetime(selected_dates))]
    if len(selected_rows) == 2:
        display(selected_rows)
    else:
        print("選んだ時刻の記録がある日を二つ指定します。")

### 3. 担当者へ結果を伝える（約 6 分）

最初の予想を振り返り，3〜4 文で報告します。次の内容を，作った表の数値と結び付けます。

1. 検証期間の全時刻での比較と，自分が選んだ時刻での比較はどうなったか。その時刻の行数と二つの MAE を添える。
2. 直線の平均残差と，選んだ二行での実測・予測から，誤差の向きや日による違いを説明する。
3. 準備にこの予測を使うか考えるため，次に比べたい記録を一つ挙げる。何をそろえ，何の違いを見るかを書く。

（ここに報告を記入）

カーネルを再起動し，上から全セルを実行します。自分が選んだ時間帯と二日分の表，報告を残したノートブックを提出します。

### データの出典

Capital Bikeshare の時間別集計は，Hadi Fanaee-T が UCI Machine Learning Repository で公開した `hour.csv` を教材用に改名したものです。 [データ配布元](https://doi.org/10.24432/C5W894) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)